In [0]:
%sql
CREATE OR REPLACE TABLE customer_dimension
USING DELTA
AS
SELECT
    customer_id,
    customer_name,
    email,
    city,
    country,
    created_date,
    load_timestamp,
    data_source,
    TO_DATE(created_date) AS effective_date,
    CAST(NULL AS DATE) AS end_date,
    'Y' AS is_current
FROM silver_customers;

In [0]:
%sql
SELECT *
FROM customer_dimension;

In [0]:
incremental_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("/Volumes/workspace/default/bronze/customers_incremental.csv")
)

incremental_df.createOrReplaceTempView("customers_incremental")

display(incremental_df)

In [0]:
%sql
SELECT
i.*
FROM customers_incremental i
JOIN customer_dimension d
ON i.customer_id = d.customer_id
WHERE d.is_current='Y'
AND
(
i.customer_name <> d.customer_name
OR
i.email <> d.email
OR
i.city <> d.city
OR
i.country <> d.country
);

In [0]:
%sql
UPDATE customer_dimension
SET
end_date = current_date(),
is_current = 'N'
WHERE customer_id IN
(
SELECT i.customer_id
FROM customers_incremental i
JOIN customer_dimension d
ON i.customer_id = d.customer_id
WHERE d.is_current='Y'
AND
(
i.customer_name <> d.customer_name
OR
i.email <> d.email
OR
i.city <> d.city
OR
i.country <> d.country
)
)
AND is_current='Y';

In [0]:
%sql
INSERT INTO customer_dimension
SELECT
i.customer_id,
i.customer_name,
i.email,
i.city,
i.country,
i.created_date,
current_timestamp(),
'customers_incremental.csv',
current_date(),
NULL,
'Y'
FROM customers_incremental i
LEFT JOIN customer_dimension d
ON i.customer_id = d.customer_id
AND d.is_current='Y'
WHERE
d.customer_id IS NULL
OR
(
i.customer_name <> d.customer_name
OR
i.email <> d.email
OR
i.city <> d.city
OR
i.country <> d.country
);

In [0]:
%sql
SELECT
customer_id,
customer_name,
city,
effective_date,
end_date,
is_current
FROM customer_dimension
WHERE customer_id IN (1003,1101)
ORDER BY customer_id,effective_date;